In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pass_at_k import pass_at_k_rates_with_sample_variance
import math
from pass_at_k import (
    BetaBinomialPassAtK,
    bootstrap_pass_at_k_ci,
)
import numpy as np


def compute(data, k_values, budget_per_problem, random_state=42):
    """Split trials per problem: first `budget` for estimators, rest for MVUE."""
    rng = np.random.RandomState(random_state)
    n_problems, n_samples = data.shape
    b = min(int(budget_per_problem), n_samples // 2)
    if b <= 0:
        raise ValueError(
            f"budget_per_problem must be in [1, {n_samples // 2}]; got {budget_per_problem}"
        )

    data_est = []
    data_mvue = []
    for i in range(n_problems):
        gens = data[i]
        perm = rng.permutation(n_samples)
        gens = gens[perm]
        data_est.append(gens[:b])
        data_mvue.append(gens[b:])
    data_mvue = np.asarray(data_mvue)

    k_values = np.asarray(k_values, dtype=int)
    max_k = data_mvue.shape[1]
    k_values = np.unique(k_values[(k_values >= 1) & (k_values <= max_k)])
    if k_values.size == 0:
        raise ValueError(f"No k values in [1, {max_k}]")

    data = {
        "successes": np.array([gens.sum(dtype=int) for gens in data_est]),
        "attempts": np.full(n_problems, b, dtype=int),
    }

    mvue_pass_at_k, mvue_pass_at_k_var = pass_at_k_rates_with_sample_variance(
        data_mvue, k_values
    )

    est = BetaBinomialPassAtK(verbose=False)

    est.fit(data['successes'], data['attempts'])
    est_int = est.predict(k_values, method="integrated", bias_correct=False)
    est_posterior = est.predict(k_values, method="posterior")

    # est_npmle = NPMLEBinomialPassAtK(verbose=False)
    # est_npmle.fit(data['successes'], data['attempts'])
    # est_npmle_pass = est_npmle.predict(k_values)

    # est_npmle_plugin = est_npmle.predict(k_values, method="plugin", bias_correct=False)
    # est_npmle_plugin_bc = est_npmle.predict(k_values, method="plugin", bias_correct=True)

    # est_beta_mixture_npmle = BetaMixtureNPMLEPassAtK(verbose=False, nu=8.0)
    # est_beta_mixture_npmle.fit(data['successes'], data['attempts'])
    # est_beta_mixture_npmle_integrated = est_beta_mixture_npmle.predict(k_values, method='integrated')
    # est_beta_mixture_npmle_posterior = est_beta_mixture_npmle.predict(k_values, method='posterior')

    # est_npmle_reg = NPMLEBinomialPassAtK(verbose=False, reg_alpha=0.001)
    # est_npmle_reg.fit(data['successes'], data['attempts'])
    # est_npmle_reg_pass = est_npmle_reg.predict(k_values)

    # est_npmle_reg_plugin = est_npmle_reg.predict(k_values, method="plugin", bias_correct=False)
    # est_npmle_reg_plugin_bc = est_npmle_reg.predict(k_values, method="plugin", bias_correct=True)

    boot_int, lower_int, upper_int = bootstrap_pass_at_k_ci(
        lambda: BetaBinomialPassAtK(verbose=False),
        data['successes'], data['attempts'],
        k_values, n_bootstraps=200,
        predict_configs=[{"method": "integrated"}]
    )
    fit = {
        "k_values": k_values,
        "estimate": est_int,
        "estimate_boot": boot_int,
        "estimate_posterior": est_posterior,
        "mvue": mvue_pass_at_k,
        "mvue_var": mvue_pass_at_k_var,
        "lower_ci": lower_int,
        "upper_ci": upper_int,
        # "npmle": est_npmle_pass,
        # "beta_mixture_integrated": est_beta_mixture_npmle_integrated,
        # "beta_mixture_posterior": est_beta_mixture_npmle_posterior,
        # # "efron_g": est_efron_g_pass,
        # "npmle_plugin": est_npmle_plugin,
        # "npmle_plugin_bc": est_npmle_plugin_bc,
        # "npmle_reg": est_npmle_reg_pass,
        # "npmle_reg_plugin": est_npmle_reg_plugin,
        # "npmle_reg_plugin_bc": est_npmle_reg_plugin_bc,
    }

    return fit


In [3]:
import math
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

import pickle
with open('r1_reasoning.pkl', 'rb') as f:
    data_dict = pickle.load(f)

# Reasoning sets: 128 samples/prompt (monkey_business in compare_real_v2 uses 10000)
SAMPLES_PER_PROMPT = 128
budget_per_problem = 10
MVUE_SAMPLES = SAMPLES_PER_PROMPT - budget_per_problem

# Log-spaced k grid (1 <= k <= MVUE_SAMPLES for unbiased pass@k on holdout fold)
k_values = np.unique(
    np.round(np.logspace(0, np.log10(MVUE_SAMPLES), 15)).astype(int)
)
k_values = k_values[(k_values >= 1) & (k_values <= MVUE_SAMPLES)]

cfg_to_data = data_dict

cfg_names = list(cfg_to_data.keys())
n_cfg = len(cfg_names)

# Precompute results per config
results = {}
for cfg in cfg_names:
    oracle_data = cfg_to_data[cfg]["data"]
    if oracle_data.shape[1] != SAMPLES_PER_PROMPT:
        raise ValueError(
            f"{cfg}: expected {SAMPLES_PER_PROMPT} samples, got {oracle_data.shape[1]}"
        )
    fit = compute(oracle_data, k_values, budget_per_problem)

    successes_oracle = np.sum(oracle_data, axis=1)
    attempts_oracle = np.full(oracle_data.shape[0], oracle_data.shape[1], dtype=np.int64)
    est_oracle = BetaBinomialPassAtK(verbose=False).fit(successes_oracle, attempts_oracle)
    alpha_oracle, beta_oracle = est_oracle.alpha_, est_oracle.beta_

    p_hat = successes_oracle / attempts_oracle.astype(float)
    p_hat_pos = p_hat[p_hat > 0]

    results[cfg] = {
        "fit": fit,
        "alpha": alpha_oracle,
        "beta": beta_oracle,
        "p_hat": p_hat,
        "p_hat_pos": p_hat_pos,
        "n_samples": oracle_data.shape[1],
    }


/home/dkp45/miniconda3/envs/compute/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Notes (128-sample reasoning data)

Compared to `compare_real_v2.ipynb` (10k trials/problem): `budget_per_problem` trials fit estimators; the rest evaluate MVUE. Pass@k uses a **log-spaced** grid from `k=1` to `k=MVUE_SAMPLES`. Prior panels use bins aligned to discrete rates `k/128` and a separate y-axis for the Beta PDF.

In [ ]:
# One big figure with n_cfg rows and 2 columns (20 axes if n_cfg=10)
fig, axes = plt.subplots(n_cfg, 2, figsize=(10, max(6, 2.0 * n_cfg)), sharex="col")

# If only one config, make axes 2D-compatible
if n_cfg == 1:
    axes = np.array([axes])

p_edges = np.arange(0, SAMPLES_PER_PROMPT + 1) / SAMPLES_PER_PROMPT
p_plot = np.linspace(1 / SAMPLES_PER_PROMPT, 1.0, 300)

for row, cfg in enumerate(cfg_names):
    res = results[cfg]
    fit = res["fit"]
    k_plot = np.asarray(fit["k_values"])
    p_hat_pos = res["p_hat_pos"]

    ax_prior = axes[row, 0]
    ax_est = axes[row, 1]

    # Left: empirical p_hat (discrete k/128) + Beta prior on separate y scale
    ax_prior.hist(
        p_hat_pos,
        # bins=p_edges,
        bins=np.logspace(-4, 0, 21),
        density=True,
        alpha=0.5,
        color="C0",
        edgecolor="white",
    )
    ax_pdf = ax_prior.twinx()
    ax_pdf.plot(
        p_plot,
        beta_dist.pdf(p_plot, res["alpha"], res["beta"]),
        "k-",
        linewidth=1.5,
    )
    ax_prior.set_xscale("log")
    ax_prior.set_xlim(1 / SAMPLES_PER_PROMPT, 1)
    ymax = ax_prior.get_ylim()[1]
    ax_prior.set_ylim(0, ymax * 1.1)
    if row == n_cfg - 1:
        ax_prior.set_xlabel("Success probability p")
    ax_prior.set_ylabel("Empirical density")
    ax_pdf.set_ylabel("Beta pdf")
    ax_prior.set_title(f"{cfg} prior", fontsize=8)

    # Right: estimators vs MVUE
    ax_est.fill_between(
        k_plot,
        fit["lower_ci"],
        fit["upper_ci"],
        alpha=0.2,
        color="C1",
    )
    estimator_specs = [
        ("estimate", "Integrated", "s-", "C1"),
        ("estimate_posterior", "Posterior", "^-", "C2"),
        ("mvue", "MVUE", "o-", "C0"),
        # ("npmle", "NPMLE", "^-", "C2"),
        # ("npmle_mixture", "NPMLE Mixture", "v-", "C3"),
        # # ("efron_g", "Efron G", "^-", "C4"),
        # ("npmle_plugin", "NPMLE Plugin", "o-", "C5"),
        # ("npmle_plugin_bc", "NPMLE Plugin (bias-corrected)", "v-", "C6"),
        # ("npmle_reg", "NPMLE Reg", "^-", "C7"),
        # ("npmle_reg_plugin", "NPMLE Reg Plugin", "o-", "C8"),
        # ("npmle_reg_plugin_bc", "NPMLE Reg Plugin (bias-corrected)", "v-", "C9"),
    ]

    for key, label, style, color in estimator_specs:
        if key in fit:
            ax_est.plot(
                k_plot,
                fit[key],
                style,
                color=color,
                linewidth=1.5,
                markersize=4,
                label=label,
            )
    
    
    
    
    
    ax_est.set_xscale("log")
    ax_est.set_xlim(k_plot.min() * 0.9, k_plot.max() * 1.05)
    if row == n_cfg - 1:
        ax_est.set_xlabel("k (number of samples)")
    ax_est.set_ylabel("pass@k")
    ax_est.grid(True, alpha=0.2)
    ax_est.set_title(f"{cfg} estimators", fontsize=8)

# Only put legend on the first estimator axis to keep things readable
first_est_ax = axes[0, 1]
handles, labels = first_est_ax.get_legend_handles_labels()
first_est_ax.legend(handles, labels, fontsize=7, loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

cfg_to_data = data_dict

# estimator_name -> {cfg -> WRMSE}
estimator_errors = {}

for cfg, oracle_data in cfg_to_data.items():
    oracle_data = oracle_data['data']
    # 1) Estimator curves on sampled data
    # fit = compute(oracle_data, k_values, total_budget, sampler="uniform")
    fit = results[cfg]['fit']
    mvue_curve = np.asarray(fit["mvue"], dtype=float)
    variance_curve = np.asarray(fit["mvue_var"], dtype=float)
    # # 2) Inverse-variance weighting curve from oracle p_true
    # successes_oracle = np.sum(oracle_data, axis=1)
    # attempts_oracle = np.full(oracle_data.shape[0], oracle_data.shape[1], dtype=np.int64)
    # p_true = successes_oracle / attempts_oracle.astype(float)

    # n_tasks = len(p_true)
    # m_samples_per_task = total_budget / n_tasks

    # k_vec = k_values[:, None]
    # p_vec = p_true[None, :]
    # variance_curve = (k_vec**2 / (n_tasks**2 * m_samples_per_task)) * np.sum(
    #     p_vec * (1.0 - p_vec) ** (2 * k_vec - 1),
    #     axis=1,
    # )
    # variance_curve = variance_curve + 1e-12

    # 3) WRMSE for each estimator in fit (except MVUE itself)
    for est_name, est_curve in fit.items():
        if est_name in ["mvue", "lower_ci", "upper_ci", "estimate_boot", "k_values"]:
            continue
        est_curve = np.asarray(est_curve, dtype=float)
        # wmse = np.mean(((est_curve - mvue_curve) ** 2) / variance_curve)
        wmse = np.mean(((est_curve - mvue_curve) ** 2))
        wrmse = float(np.sqrt(wmse))
        estimator_errors.setdefault(est_name, {})[cfg] = wrmse

# ---------- Reporting (clean aligned table) ----------
error_df = pd.DataFrame(estimator_errors).sort_index()
error_df.index.name = "config"

# Pretty column names
error_df = error_df.rename(columns={c: f"WRMSE[{c}]" for c in error_df.columns})

# Per-config table
display(error_df.style.format("{:.4f}"))

# Summary table (mean/std across configs)
summary_df = pd.DataFrame({
    "mean": error_df.mean(axis=0),
    "std": error_df.std(axis=0),
    "min": error_df.min(axis=0),
    "max": error_df.max(axis=0),
}).sort_values("mean")

display(summary_df.style.format("{:.4f}"))

# If you need plain text output too:
print("\nPer-config WRMSE table:\n")
print(error_df.to_string(float_format=lambda x: f"{x:.4f}"))

print("\nEstimator summary (sorted by mean WRMSE):\n")
print(summary_df.to_string(float_format=lambda x: f"{x:.4f}"))

,WRMSE[estimate],WRMSE[estimate_posterior],WRMSE[mvue_var]
config,,,
aime25_qwen3-14b,0.0560,0.0473,0.7334
aime25_qwen3-8b,0.0422,0.0387,0.6753
aime25_r1-qwen14b,0.0460,0.0471,0.6895
aime25_r1-qwen7b,0.0258,0.0244,0.5870
gpqa-diamond_qwen3-14b,0.0068,0.0054,0.7966
gpqa-diamond_qwen3-8b,0.0092,0.0084,0.7865
gpqa-diamond_r1-qwen14b,0.0078,0.0046,0.7608
gpqa-diamond_r1-qwen7b,0.0708,0.0657,0.7313
hmmtfeb25_qwen3-14b,0.0696,0.0697,0.5613


,mean,std,min,max
WRMSE[estimate_posterior],0.0383,0.0278,0.0046,0.1123
WRMSE[estimate],0.0406,0.0285,0.0068,0.1161
WRMSE[mvue_var],0.6147,0.1245,0.4469,0.7966



Per-config WRMSE table:

                                    WRMSE[estimate]  WRMSE[estimate_posterior]  WRMSE[mvue_var]
config                                                                                         
aime25_qwen3-14b                             0.0560                     0.0473           0.7334
aime25_qwen3-8b                              0.0422                     0.0387           0.6753
aime25_r1-qwen14b                            0.0460                     0.0471           0.6895
aime25_r1-qwen7b                             0.0258                     0.0244           0.5870
gpqa-diamond_qwen3-14b                       0.0068                     0.0054           0.7966
gpqa-diamond_qwen3-8b                        0.0092                     0.0084           0.7865
gpqa-diamond_r1-qwen14b                      0.0078                     0.0046           0.7608
gpqa-diamond_r1-qwen7b                       0.0708                     0.0657           0.7313
hmmtfeb25_qwen